In [ ]:
from google.colab import drive
drive.mount('/content/drive')
curr_dir =  "/content/drive/MyDrive/SEGMENTATION_QAZ"
%cd "$curr_dir"

Mounted at /content/drive
/content/drive/MyDrive/SEGMENTATION_QAZ


In [ ]:
pip install xlwt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.0/100.0 kB 3.6 MB/s eta 0:00:00


In [ ]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.4 MB/s eta 0:00:00


In [ ]:
import re
import os
import xlrd, xlwt
import sacrebleu
import numpy as np
import openpyxl
import time
from tqdm import tqdm

def splitting_by_words(text):
    result = re.findall(r'\w+|[^\w\s]', text)
    print('text', result)
    return result


ending_segs = []
def sorting_endings(endings_file_name):
    endings_wb = xlrd.open_workbook(endings_file_name)
    endings_sh = endings_wb.sheet_by_index(0)
    endings = []
    ending_segs = []
    for rownum in range(endings_sh.nrows-1):
        ending = endings_sh.cell(rownum+1,0).value
        if '\ufeff' in ending:
            ending = ending.replace('\ufeff', '')
        endings.append(ending)
    #print('''ENDINGS''')
    #print(endings)
    for rownum in range(endings_sh.nrows-1):
        ending_seg1 = endings_sh.cell(rownum+1,1).value
        if '\ufeff' in ending:
            ending_seg1 = ending.replace('\ufeff', '')
        ending_segs.append(ending_seg1)

    return [endings, ending_segs]




def f_stem(word):
    rez_stem = ''

    word_len = len(word)
    min_len_of_stem = 2
    word_len = len(word)
    if word_len > min_len_of_stem:

        n = word_len - min_len_of_stem
        i = 1  # Начинаем с 1, если индексировать с конца слова
        #maxlenjurnaq = 4  # Максимальная длина суффикса (журнака)

        while i <= len(word):
            word_ending = word[-i:]  # Получаем окончание длиной i
            #print('jurnaq', word_ending)
            stem = word[:-i]  # Основная часть слова без окончания
            #print('stem', stem)

            i += 1  # Увеличиваем i на каждой итерации
    else:
        rez_stem = word

    #print('stem', rez_stem)
    #print('jurnaq1', jurnaq_stem, rez_stem)
    return rez_stem


def ending_stem(word):
    word_len = len(word)
    min_len_of_word = 2
    rez_stem = ''
    rez_ending = ''
    if word_len > min_len_of_word:
        n = word_len - min_len_of_word
        if word.lower() in stems_list:
            rez_stem = word
            rez_ending = ''
            #print('stem 0', rez_stem)
            #print('ending 0', rez_ending)

            return rez_stem, rez_ending

        else:
            i = n+1
            while i > 0:
                word_ending = word[word_len - (i-1):]
                stem = word[:word_len-len(word_ending)]
                #print('stem', stem)
                #print('ending', word_ending)
                for ending in endings:
                    if word_ending == ending:
                        if stem.lower() in stems_list:
                            rez_stem = stem
                            rez_ending = word_ending
                            #print('stem 2', rez_stem)
                            #print('ending 2', rez_ending)
                            i = 0
                            break

                #print('stem 1', stem)

                rez_stem = stem
                i = i-1
    else:
        rez_stem = word
        rez_ending = ''
    return rez_stem, rez_ending


global endings_seg

def segment_word(word, endings):

    ending_text = {}
    ending_seg_dic = {}
    endingseg = ''
    endingseg_1 = ''
    #print('word', word)
    stemm, ending = ending_stem(word)                 # вызов функции сегментирования слова на стем и окончание
    #print('stem', stemm)
    #print('ending', ending)
    stemm_len = len(stemm)

    if ending != '':
        num_ending = endings.index(ending)           # находится индекс окончание
            #print(num_ending)
        endingseg = endings_seg[num_ending]          # находится сегментированное окончание
    else:
        endingseg = ''
            #endingseg = "@@ ".join([stemm, endingseg])
            #ending_seg_dic.update({word: endingseg})
        #print('stemm1' ,stemm)                                                  # СЕГМЕНТИРОВАНИЕ СТЕМА ПО ЖУРНАКАМ

        # Формируем финальную строку: Стем + журнаки + окончания
    #print('stemm', stemm)
    #print('endingseg', endingseg)
    #print('jurnaqs_seg', jurnaqs_seg)
    if endingseg != '':
        final_seg = f"{stemm}@@ {endingseg}"
        #print('final_seg1', final_seg)

    else:
        final_seg = f"{stemm}"
        #print('final_seg4', final_seg)


    return final_seg

def morpho_segment(text):
    tokens = re.findall(r'\w+|[^\w\s]', text)  # Слова + пунктуация
    processed_tokens = []
    for token in tokens:
        if token.lower() in stop_words:
            processed_tokens.append(token)  # Оставляем стоп-слова без изменений
        elif token.isalpha():
            #print('token', token)
            processed_tokens.append(segment_word(token, endings))  # Сегментация слов
        else:
            processed_tokens.append(token)  # Оставляем числа и знаки препинания

    return " ".join(processed_tokens)

stopwords_file_name = "stop_words.txt"
stems_file_name = "qaz_stems_unik_edit.xlsx"
#jurnaqs_file_name = "qaz_jurnaks.xls"
endings_file_name = "qaz_endings_seg.xls"
endings_seg = []
endings, endings_seg = sorting_endings(endings_file_name)
#jurnaqs = sorting_jurnaqs(jurnaqs_file_name)

        #------INPUT STEM FILE
stems_wb = openpyxl.load_workbook(stems_file_name)
stems_sh = stems_wb.active
stems = []
global stems_list
for row in stems_sh.rows:
    stem = row[0].value
    if '\ufeff' in stem:
        stem = stem.replace('\ufeff', '')
    stems.append(stem)
stems_list = sorted(stems, key=len, reverse=True)

         #-----INPUT STOPWORD FILE
with open(stopwords_file_name, "r", encoding="utf-8") as f:
    stopwords_file = f.readlines()
    stop_words = []
    for stop_word in stopwords_file:
        if "\n" in stop_word:
            stop_word = stop_word.replace("\n", "")
        stop_words.append(stop_word)
    #print(stop_words)


input_path = "kaz_285000_only_284707.txt"
output_dir = "segmented_qaz_284707"
chunk_size = 5000


# Читаем строки
with open(input_path, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

os.makedirs(output_dir, exist_ok=True)

n_chunks = (len(lines) + chunk_size - 1) // chunk_size
print(f"📄 Всего чанков: {n_chunks}")
                  #------ END INPUT TEXT FILE AND CHANKING

for i in tqdm(range(n_chunks), desc="⏳ Сегментирую"):
    # Путь для сохранённого чанка
    output_path = os.path.join(output_dir, f"chunk_{i:03d}.txt")

    # Проверяем: если файл уже существует, пропускаем
    if os.path.exists(output_path):
        print(f"⚡ Чанк {i} уже существует, пропускаем...")
        continue

    # Загружаем нужные строки
    chunk_lines = lines[i * chunk_size: (i + 1) * chunk_size]

    start_time = time.time()  # Засекаем время

    # Сегментация
    segmented = [morpho_segment(line) for line in chunk_lines]

    # Сохраняем результат
    with open(output_path, "w", encoding="utf-8") as f_out:
        for line in segmented:
            f_out.write(line + "\n")

    # Логгируем время
    duration = time.time() - start_time
    print(f"✅ Чанк {i} обработан за {duration:.2f} сек")





📄 Всего чанков: 57


⏳ Сегментирую:   2%|▏         | 1/57 [04:57<4:37:35, 297.42s/it]

✅ Чанк 0 обработан за 297.42 сек


⏳ Сегментирую:   4%|▎         | 2/57 [09:49<4:29:33, 294.05s/it]

✅ Чанк 1 обработан за 291.70 сек


...

✅ Чанк 54 обработан за 412.23 сек


⏳ Сегментирую:  98%|█████████▊| 56/57 [6:10:38<06:06, 366.97s/it]

✅ Чанк 55 обработан за 274.16 сек


⏳ Сегментирую: 100%|██████████| 57/57 [6:14:16<00:00, 393.97s/it]

✅ Чанк 56 обработан за 217.38 сек
